In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt



In [2]:
DB_PATH = "../f1.db"

TABLES = [
    "dim_driver", "dim_team", "dim_circuit", "dim_tyre_compound",
    "dim_race", "fact_laps", "fact_race_results", "fact_weather"
]

conn = sqlite3.connect(DB_PATH)
dfs = {t: pd.read_sql(f"SELECT * FROM {t}", conn) for t in TABLES}
conn.close()

dfs["fact_laps"]

,lap_id,race_id,driver_id,team_id,compound_id,lap_number,lap_time,tyre_age,stint_number,track_position,pit_in_time,pit_out_time
0,188483,53,44,20,1.0,1,0 days 00:01:27.905000,1.0,1.0,2.0,None,None
1,188484,53,44,20,1.0,2,0 days 00:01:26.163000,2.0,1.0,2.0,None,None
2,188485,53,44,20,1.0,3,0 days 00:01:25.702000,3.0,1.0,2.0,None,None
3,188486,53,44,20,1.0,4,0 days 00:01:26.050000,4.0,1.0,2.0,None,None
4,188487,53,44,20,1.0,5,0 days 00:01:26.176000,5.0,1.0,2.0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
188477,376960,58,55,24,1.0,67,0 days 00:01:23.238000,36.0,3.0,8.0,None,None
188478,376961,58,55,24,1.0,68,0 days 00:01:23.362000,37.0,3.0,8.0,None,None
188479,376962,58,55,24,1.0,69,0 days 00:01:23.168000,38.0,3.0,8.0,None,None
188480,376963,58,55,24,1.0,70,0 days 00:01:22.980000,39.0,3.0,8.0,None,None


In [3]:
laps = dfs["fact_laps"]

print(laps[["lap_time", "pit_in_time", "pit_out_time"]].dtypes)
print()

pit_in_filled = laps["pit_in_time"].notna().sum()
pit_out_filled = laps["pit_out_time"].notna().sum()
print(f"pit_in_time filled: {pit_in_filled} out of {len(laps)}")
print(f"pit_out_time filled: {pit_out_filled} out of {len(laps)}")
print()

sample_pit = laps[laps["pit_in_time"].notna()][["lap_id", "lap_time", "pit_in_time", "pit_out_time"]].head()
print(sample_pit)

lap_time        object
pit_in_time     object
pit_out_time    object
dtype: object

pit_in_time filled: 6155 out of 188482
pit_out_time filled: 6069 out of 188482

     lap_id                lap_time             pit_in_time pit_out_time
19   188502  0 days 00:01:30.849000  0 days 01:51:13.003000         None
60   188543  0 days 00:01:32.034000  0 days 01:37:06.997000         None
81   188564  0 days 00:01:32.204000  0 days 02:08:08.289000         None
122  188605  0 days 00:01:30.934000  0 days 01:52:46.112000         None
173  188656  0 days 00:01:32.377000  0 days 01:53:02.971000         None


In [4]:
check = laps[(laps["driver_id"] == 44) & (laps["race_id"] == 53)].sort_values("lap_number")
check = check[["lap_number", "lap_time", "pit_in_time", "pit_out_time"]]

pd.set_option("display.max_rows", 30)
print(check[(check["lap_number"] >= 15) & (check["lap_number"] <= 25)])

    lap_number                lap_time             pit_in_time  \
14          15  0 days 00:01:26.782000                    None   
15          16  0 days 00:01:26.718000                    None   
16          17  0 days 00:01:26.733000                    None   
17          18  0 days 00:01:26.273000                    None   
18          19  0 days 00:01:26.438000                    None   
19          20  0 days 00:01:30.849000  0 days 01:51:13.003000   
20          21  0 days 00:01:45.660000                    None   
21          22  0 days 00:01:25.885000                    None   
22          23  0 days 00:01:26.159000                    None   
23          24  0 days 00:01:26.327000                    None   
24          25  0 days 00:01:25.914000                    None   

              pit_out_time  
14                    None  
15                    None  
16                    None  
17                    None  
18                    None  
19                    None  
20  

In [5]:
both_filled = laps[(laps["pit_in_time"].notna()) & (laps["pit_out_time"].notna())]
only_in = laps[(laps["pit_in_time"].notna()) & (laps["pit_out_time"].isna())]
only_out = laps[(laps["pit_in_time"].isna()) & (laps["pit_out_time"].notna())]

print(f"both filled on same row: {len(both_filled)}")
print(f"only pit_in_time filled: {len(only_in)}")
print(f"only pit_out_time filled: {len(only_out)}")


both filled on same row: 323
only pit_in_time filled: 5832
only pit_out_time filled: 5746


In [6]:
check_retirement = only_in.merge(
    laps.groupby(["driver_id", "race_id"])["lap_number"].max().reset_index().rename(columns={"lap_number": "max_lap"}),
    on=["driver_id", "race_id"]
)

check_retirement["is_last_lap"] = check_retirement["lap_number"] == check_retirement["max_lap"]
print(check_retirement["is_last_lap"].value_counts())

is_last_lap
False    5614
True      218
Name: count, dtype: int64


In [7]:
laps_sorted = laps.sort_values(["driver_id", "race_id", "lap_number"]).copy()

laps_sorted["next_pit_out"] = laps_sorted.groupby(["driver_id", "race_id"])["pit_out_time"].shift(-1)
laps_sorted["next_lap_number"] = laps_sorted.groupby(["driver_id", "race_id"])["lap_number"].shift(-1)

in_lap_only = laps_sorted["pit_in_time"].notna() & laps_sorted["pit_out_time"].isna()

is_adjacent = laps_sorted["next_lap_number"] == laps_sorted["lap_number"] + 1
has_next_out = laps_sorted["next_pit_out"].notna()

matched = in_lap_only & is_adjacent & has_next_out
unmatched = in_lap_only & ~(is_adjacent & has_next_out)

print(f"in-laps matched to next row out-lap: {matched.sum()}")
print(f"in-laps not matched: {unmatched.sum()}")

in-laps matched to next row out-lap: 5613
in-laps not matched: 219


In [8]:
laps["lap_seconds"] = pd.to_timedelta(laps["lap_time"], errors="coerce").dt.total_seconds()

print(laps["lap_seconds"].describe())
print()
print(f"laps at or below zero seconds: {(laps['lap_seconds'] <= 0).sum()}")
print(f"laps over 300 seconds: {(laps['lap_seconds'] > 300).sum()}")
print(f"null lap_seconds after conversion: {laps['lap_seconds'].isna().sum()}")

count    185326.000000
mean         91.473275
std          26.636737
min          55.404000
25%          80.999000
50%          89.724000
75%          99.646000
max        2526.253000
Name: lap_seconds, dtype: float64

laps at or below zero seconds: 0
laps over 300 seconds: 16
null lap_seconds after conversion: 3156


In [9]:
extreme = laps[laps["lap_seconds"] > 300][["lap_id", "race_id", "driver_id", "lap_number", "lap_time", "track_position"]]
print(extreme.sort_values("lap_time", ascending=False))
print()

null_laps = laps[laps["lap_seconds"].isna()]
print(f"races involved in null laps: {null_laps['race_id'].nunique()}")
print(null_laps[["lap_id", "race_id", "driver_id", "lap_number", "lap_time"]].head(10))

        lap_id  race_id  driver_id  lap_number                lap_time  \
129323  317806      157         57           1  0 days 00:42:06.253000   
129095  317578      157         53           1  0 days 00:42:01.748000   
129247  317730      157         56           1  0 days 00:41:56.858000   
128943  317426      157         52           1  0 days 00:41:53.184000   
129019  317502      157         70           1  0 days 00:41:47.607000   
129171  317654      157         59           1  0 days 00:41:42.648000   
128866  317349      157         58           1  0 days 00:41:37.695000   
128789  317272      157         50           1  0 days 00:41:34.817000   
128712  317195      157         63           1  0 days 00:41:28.609000   
128634  317117      157         49           1  0 days 00:41:23.804000   
128556  317039      157         44           1  0 days 00:41:18.944000   
128478  316961      157         48           1  0 days 00:41:13.884000   
128400  316883      157         51    

In [10]:
null_laps = laps[laps["lap_seconds"].isna()].copy()

max_lap_per_race = laps.groupby("race_id")["lap_number"].max().reset_index()
max_lap_per_race = max_lap_per_race.rename(columns={"lap_number": "max_lap"})

null_laps = null_laps.merge(max_lap_per_race, on="race_id")
null_laps["is_final_lap"] = null_laps["lap_number"] == null_laps["max_lap"]

print(null_laps["is_final_lap"].value_counts())
print()
print(f"distinct races affected: {null_laps['race_id'].nunique()}")

is_final_lap
False    3089
True       67
Name: count, dtype: int64

distinct races affected: 142


In [11]:
null_laps = laps[laps["lap_seconds"].isna()].copy()

driver_max_lap = laps.groupby(["driver_id", "race_id"])["lap_number"].max().reset_index()
driver_max_lap = driver_max_lap.rename(columns={"lap_number": "driver_max_lap"})

null_laps = null_laps.merge(driver_max_lap, on=["driver_id", "race_id"])
null_laps["is_driver_last_lap"] = null_laps["lap_number"] == null_laps["driver_max_lap"]

print(null_laps["is_driver_last_lap"].value_counts())

is_driver_last_lap
False    2772
True      384
Name: count, dtype: int64


In [12]:
remaining = null_laps[~null_laps["is_driver_last_lap"]]
print(f"remaining unexplained nulls: {len(remaining)}")
print()

sample = remaining[["lap_id", "race_id", "driver_id", "lap_number", "track_position", "pit_in_time", "pit_out_time"]].sample(15, random_state=1)
print(sample.sort_values(["race_id", "driver_id", "lap_number"]))

remaining unexplained nulls: 2772

      lap_id  race_id  driver_id  lap_number  track_position  \
491   211665       23         58          50             4.0   
631   216694       43         46          13             4.0   
829   233692       52         59          64            13.0   
341   203528       54         47          21             6.0   
1984  288319       63         53          17            10.0   
1713  278746       64         63          25            15.0   
1496  260808       65         75           2            17.0   
2601  324572       71         61           2            14.0   
2385  314915       75         63           3            16.0   
332   201990       84         64           3            17.0   
2123  294853       95         49          21             6.0   
1744  281326      154         70          26            10.0   
1289  250449      168         54           4            13.0   
38    190360      170         46          33            14.0   
637  

In [14]:
remaining_full = laps.loc[remaining["lap_id"]]

print(f"of the unexplained nulls, is_in_lap: {remaining['pit_in_time'].notna().sum()}")
print(f"of the unexplained nulls, is_out_lap: {remaining['pit_out_time'].notna().sum()}")
print(f"neither: {(remaining['pit_in_time'].isna() & remaining['pit_out_time'].isna()).sum()}")

KeyError: "None of [Index([189471, 189472, 189540, 189541, 189609, 189610, 189678, 189679, 189747,\n       189748,\n       ...\n       376573, 376643, 376644, 376645, 376714, 376715, 376716, 376786, 376857,\n       376928],\n      dtype='int64', length=2772)] are in the [index]"

In [15]:
print(f"of the unexplained nulls, is_in_lap: {remaining['pit_in_time'].notna().sum()}")
print(f"of the unexplained nulls, is_out_lap: {remaining['pit_out_time'].notna().sum()}")
print(f"neither: {(remaining['pit_in_time'].isna() & remaining['pit_out_time'].isna()).sum()}")

of the unexplained nulls, is_in_lap: 423
of the unexplained nulls, is_out_lap: 747
neither: 1748


In [ ]:
sample = remaining[["lap_id", "race_id", "driver_id", "lap_number", "track_position", "pit_in_time", "pit_out_time"]].sample(15, random_state=1)
print(sample.sort_values(["race_id", "driver_id", "lap_number"]))

      lap_id  race_id  driver_id  lap_number  track_position  \
491   211665       23         58          50             4.0   
631   216694       43         46          13             4.0   
829   233692       52         59          64            13.0   
341   203528       54         47          21             6.0   
1984  288319       63         53          17            10.0   
1713  278746       64         63          25            15.0   
1496  260808       65         75           2            17.0   
2601  324572       71         61           2            14.0   
2385  314915       75         63           3            16.0   
332   201990       84         64           3            17.0   
2123  294853       95         49          21             6.0   
1744  281326      154         70          26            10.0   
1289  250449      168         54           4            13.0   
38    190360      170         46          33            14.0   
637   217953      172         44        

## Data Quality Findings

### lap_time nulls (3156 rows, ~1.7% of fact_laps)
- 67 rows: race's actual final lap, session ends mid-timing, expected
- 384 rows: driver's own final lap (retirement/DNF), no time posted, expected
- ~1000 rows: overlap with pit in-lap or out-lap, timing interrupted by pit entry/exit
- ~1700 rows: no identifiable pattern (checked race_id, driver_id, lap_number,
  track_position, pit flags), likely individual FastF1 timing loop gaps
- Treatment: no special handling needed, NaN naturally excluded from any
  lap_seconds based feature

### Extreme lap times (16 rows over 300 seconds)
- All from race_id 157, lap_number 1, across ~16 different drivers
- Points to a red flag or race restart right after start, clock did not reset
- Lap 2 onward for the same race needs to be confirmed as normal
- Treatment: exclude from lap time based features using a sane upper threshold
  (likely 180-200 seconds), document as a known event tied to that race

### Pit stop timing structure
- pit_in_time and pit_out_time represent session elapsed time, not duration
- 323 pit stops: both times on the same row
- 5613 pit stops: split across two rows, in-lap then out-lap on the next lap_number
- 219 rows: pit_in_time with no matching out-lap, driver retired right after pit entry
- Treatment: pit stop duration extraction needs to handle both row patterns
  separately, this is feature engineering work, not covered in tier one

results = dfs["fact_race_results"]

filled = results["fastest_lap_time"].notna().sum()
total = len(results)
print(f"filled: {filled} out of {total}")

In [ ]:
results = dfs["fact_race_results"]

filled = results["fastest_lap_time"].notna().sum()
total = len(results)
print(f"filled: {filled} out of {total}")

filled: 0 out of 3438


In [ ]:
per_race_filled = results[results["fastest_lap_time"].notna()].groupby("race_id").size()
print(per_race_filled.value_counts())
print()
print(f"races with a fastest_lap_time recorded: {per_race_filled.shape[0]}")
print(f"total races in fact_race_results: {results['race_id'].nunique()}")

Series([], Name: count, dtype: int64)

races with a fastest_lap_time recorded: 0
total races in fact_race_results: 172


In [ ]:

races_with_results = set(dfs["fact_race_results"]["race_id"].unique())
all_races = set(dfs["dim_race"]["race_id"].unique())

missing = all_races - races_with_results
print(missing)

dfs["dim_race"][dfs["dim_race"]["race_id"].isin(missing)]

{np.int64(142)}


,race_id,year,round,race_name,circuit_id,race_date,season_era
141,142,2018,14,Italian Grand Prix,6,2018-09-02,pre_covid


In [ ]:
laps = dfs["fact_laps"]

fastest = laps.groupby(["race_id", "driver_id"])["lap_seconds"].min().reset_index()
fastest = fastest.rename(columns={"lap_seconds": "fastest_lap_seconds"})

fastest.head()

,race_id,driver_id,fastest_lap_seconds
0,1,44,69.351
1,1,45,68.305
2,1,46,67.974
3,1,47,67.901
4,1,48,69.317


In [ ]:
conn = sqlite3.connect("../f1.db")
cursor = conn.cursor()

updates = [
    (float(row.fastest_lap_seconds), int(row.race_id), int(row.driver_id))
    for row in fastest.itertuples()
    if pd.notna(row.fastest_lap_seconds)
]

cursor.executemany(
    "UPDATE fact_race_results SET fastest_lap_time = ? WHERE race_id = ? AND driver_id = ?",
    updates
)
conn.commit()
conn.close()

print(f"updated {len(updates)} rows")

updated 3350 rows


In [ ]:
conn = sqlite3.connect("../f1.db")
dfs["fact_race_results"] = pd.read_sql("SELECT * FROM fact_race_results", conn)
conn.close()

filled = dfs["fact_race_results"]["fastest_lap_time"].notna().sum()
total = len(dfs["fact_race_results"])
print(f"filled: {filled} out of {total}")

dfs["fact_race_results"].head()

filled: 3350 out of 3438


,result_id,race_id,driver_id,team_id,grid_position,finish_position,points,status,fastest_lap_time
0,3439,53,44,20,2.0,1.0,25.0,Finished,85.240
1,3440,53,45,20,5.0,2.0,18.0,Finished,85.522
2,3441,53,46,21,1.0,3.0,15.0,Finished,85.501
3,3442,53,47,21,3.0,4.0,12.0,Finished,85.580
4,3443,53,48,22,4.0,5.0,10.0,Finished,85.847


In [ ]:
results = dfs["fact_race_results"]
laps = dfs["fact_laps"]

drivers_with_laps = set(zip(laps["race_id"], laps["driver_id"]))
results["has_lap_data"] = results.apply(lambda r: (r["race_id"], r["driver_id"]) in drivers_with_laps, axis=1)

missing = results[results["fastest_lap_time"].isna()]
print(f"missing fastest_lap_time: {len(missing)}")
print()
print(missing["status"].value_counts())

missing fastest_lap_time: 88

status
Collision           32
Retired             28
Did not start        9
Accident             8
Withdrew             2
Power Unit           2
Driveshaft           1
Brakes               1
Exhaust              1
Hydraulics           1
Collision damage     1
Suspension           1
Illness              1
Name: count, dtype: int64


In [ ]:
laps = dfs["fact_laps"].copy()

laps["lap_seconds"] = pd.to_timedelta(laps["lap_time"], errors="coerce").dt.total_seconds()
laps["pit_in_seconds"] = pd.to_timedelta(laps["pit_in_time"], errors="coerce").dt.total_seconds()
laps["pit_out_seconds"] = pd.to_timedelta(laps["pit_out_time"], errors="coerce").dt.total_seconds()

laps["is_in_lap"] = laps["pit_in_time"].notna()
laps["is_out_lap"] = laps["pit_out_time"].notna()

dfs["fact_laps"] = laps
dfs["fact_laps"].head()

,lap_id,race_id,driver_id,team_id,compound_id,lap_number,lap_time,tyre_age,stint_number,track_position,pit_in_time,pit_out_time,lap_seconds,pit_in_seconds,pit_out_seconds,is_in_lap,is_out_lap
0,188483,53,44,20,1.0,1,0 days 00:01:27.905000,1.0,1.0,2.0,None,None,87.905,NaN,NaN,False,False
1,188484,53,44,20,1.0,2,0 days 00:01:26.163000,2.0,1.0,2.0,None,None,86.163,NaN,NaN,False,False
2,188485,53,44,20,1.0,3,0 days 00:01:25.702000,3.0,1.0,2.0,None,None,85.702,NaN,NaN,False,False
3,188486,53,44,20,1.0,4,0 days 00:01:26.050000,4.0,1.0,2.0,None,None,86.050,NaN,NaN,False,False
4,188487,53,44,20,1.0,5,0 days 00:01:26.176000,5.0,1.0,2.0,None,None,86.176,NaN,NaN,False,False


In [ ]:
laps = dfs["fact_laps"]

pit_rows = laps[laps["is_in_lap"] | laps["is_out_lap"]]
pit_rows[["race_id", "driver_id", "lap_number", "lap_time", "lap_seconds",
          "pit_in_time", "pit_in_seconds", "pit_out_time", "pit_out_seconds",
          "is_in_lap", "is_out_lap"]].head(10)

,race_id,driver_id,lap_number,lap_time,lap_seconds,pit_in_time,pit_in_seconds,pit_out_time,pit_out_seconds,is_in_lap,is_out_lap
19,53,44,20,0 days 00:01:30.849000,90.849,0 days 01:51:13.003000,6673.003,None,NaN,True,False
20,53,44,21,0 days 00:01:45.660000,105.660,None,NaN,0 days 01:51:37.284000,6697.284,False,True
60,53,58,10,0 days 00:01:32.034000,92.034,0 days 01:37:06.997000,5826.997,None,NaN,True,False
61,53,58,11,0 days 00:01:50.170000,110.170,None,NaN,0 days 01:37:34.372000,5854.372,False,True
81,53,58,31,0 days 00:01:32.204000,92.204,0 days 02:08:08.289000,7688.289,None,NaN,True,False
82,53,58,32,0 days 00:01:46.720000,106.720,None,NaN,0 days 02:08:32.961000,7712.961,False,True
122,53,45,21,0 days 00:01:30.934000,90.934,0 days 01:52:46.112000,6766.112,None,NaN,True,False
123,53,45,22,0 days 00:01:45.228000,105.228,None,NaN,0 days 01:53:09.955000,6789.955,False,True
173,53,52,21,0 days 00:01:32.377000,92.377,0 days 01:53:02.971000,6782.971,None,NaN,True,False
174,53,52,22,0 days 00:01:47.272000,107.272,None,NaN,0 days 01:53:27.799000,6807.799,False,True


In [ ]:
laps = dfs["fact_laps"].sort_values(["driver_id", "race_id", "lap_number"]).copy()

laps["next_pit_out_seconds"] = laps.groupby(["driver_id", "race_id"])["pit_out_seconds"].shift(-1)
laps["next_lap_number"] = laps.groupby(["driver_id", "race_id"])["lap_number"].shift(-1)

same_row = laps[laps["pit_in_seconds"].notna() & laps["pit_out_seconds"].notna()].copy()
same_row["pit_stop_duration"] = same_row["pit_out_seconds"] - same_row["pit_in_seconds"]

split_row = laps[laps["pit_in_seconds"].notna() & laps["pit_out_seconds"].isna()].copy()
split_row = split_row[split_row["next_lap_number"] == split_row["lap_number"] + 1]
split_row["pit_stop_duration"] = split_row["next_pit_out_seconds"] - split_row["pit_in_seconds"]

pit_stops = pd.concat([
    same_row[["race_id", "driver_id", "lap_number", "pit_stop_duration"]],
    split_row[["race_id", "driver_id", "lap_number", "pit_stop_duration"]]
]).sort_values(["race_id", "driver_id", "lap_number"]).reset_index(drop=True)

pit_stops

,race_id,driver_id,lap_number,pit_stop_duration
0,1,45,21,22.379
1,1,45,51,16.921
2,1,46,21,23.828
3,1,46,47,21.825
4,1,46,51,19.947
...,...,...,...,...
5932,173,61,30,-100.411
5933,173,61,55,22.032
5934,173,63,22,21.809
5935,173,65,3,24.952


In [ ]:
print(pit_stops["pit_stop_duration"].describe())
print()
print(f"total pit stops found: {len(pit_stops)}")

count    5936.000000
mean      125.019212
std       596.065958
min      -188.979000
25%        21.751500
50%        23.584500
75%        27.531000
max      7610.050000
Name: pit_stop_duration, dtype: float64

total pit stops found: 5937


In [ ]:
import glob

files = glob.glob("../data/raw/race_control/*.csv")
print(f"total race control files: {len(files)}")

bad_files = []
sample_dfs = []

for f in files:
    try:
        df = pd.read_csv(f)
        if df.empty:
            bad_files.append((f, "empty"))
            continue
        df["source_file"] = f
        sample_dfs.append(df)
    except Exception as e:
        bad_files.append((f, str(e)))

print(f"good files: {len(sample_dfs)}")
print(f"bad or empty files: {len(bad_files)}")
print()
for f, reason in bad_files[:10]:
    print(f, reason)

total race control files: 173
good files: 173
bad or empty files: 0



In [ ]:
race_control_all = pd.concat(sample_dfs, ignore_index=True)
print(f"total rows across all files: {len(race_control_all)}")
print()
print("Category values:")
print(race_control_all["Category"].value_counts())
print()
print("Flag values:")
print(race_control_all["Flag"].value_counts())
print()
print("Status values:")
print(race_control_all["Status"].value_counts())


total rows across all files: 14992

Category values:
Category
Flag         8873
Other        4906
Drs           524
SafetyCar     443
CarEvent      246
Name: count, dtype: int64

Flag values:
Flag
BLUE               5388
CLEAR              1547
YELLOW              717
DOUBLE YELLOW       621
GREEN               278
CHEQUERED           171
BLACK AND WHITE     124
RED                  27
Name: count, dtype: int64

Status values:
Status
ENABLED                 322
DEPLOYED                231
DISABLED                202
IN THIS LAP             116
ENDING                   82
THROUGH THE PIT LANE     14
Name: count, dtype: int64


In [ ]:
print(race_control_all["Status"].value_counts())



Status
ENABLED                 322
DEPLOYED                231
DISABLED                202
IN THIS LAP             116
ENDING                   82
THROUGH THE PIT LANE     14
Name: count, dtype: int64


In [ ]:

race_157_control = race_control_all[race_control_all["source_file"].str.contains("157")]
print(race_157_control[["Category", "Message", "Flag", "Status", "Lap"]].to_string())

Empty DataFrame
Columns: [Category, Message, Flag, Status, Lap]
Index: []


In [ ]:
race_info = dfs["dim_race"][dfs["dim_race"]["race_id"] == 157]
print(race_info[["race_id", "year", "round", "race_name"]])


year = race_info["year"].values[0]
round_num = race_info["round"].values[0]

file_path = f"../data/raw/race_control/{year}_{round_num}.csv"
race_157_control = pd.read_csv(file_path)

print(race_157_control[["Category", "Message", "Flag", "Status", "Lap"]].to_string())

     race_id  year  round          race_name
156      157  2024      8  Monaco Grand Prix
    Category                                                                                                                    Message           Flag    Status  Lap
0       Flag                                                                                                GREEN LIGHT - PIT EXIT OPEN          GREEN       NaN    1
1      Other                                                                                                            PIT EXIT CLOSED            NaN       NaN    1
2       Flag                                                                                            DOUBLE YELLOW IN TRACK SECTOR 7  DOUBLE YELLOW       NaN    1
3       Flag                                                                                                    CLEAR IN TRACK SECTOR 7          CLEAR       NaN    1
4       Flag                                                                    

In [ ]:
red_flag_rows = race_157_control[race_157_control["Flag"] == "RED"]
print(red_flag_rows[["Category", "Message", "Flag", "Status", "Lap"]].to_string())

   Category   Message Flag Status  Lap
22     Flag  RED FLAG  RED    NaN    1


In [ ]:
early_race = race_157_control[race_157_control["Lap"] <= 2]
print(early_race[["Category", "Message", "Flag", "Status", "Lap"]].to_string())

   Category                                                                                                       Message           Flag    Status  Lap
0      Flag                                                                                   GREEN LIGHT - PIT EXIT OPEN          GREEN       NaN    1
1     Other                                                                                               PIT EXIT CLOSED            NaN       NaN    1
2      Flag                                                                               DOUBLE YELLOW IN TRACK SECTOR 7  DOUBLE YELLOW       NaN    1
3      Flag                                                                                       CLEAR IN TRACK SECTOR 7          CLEAR       NaN    1
4      Flag                                                                              DOUBLE YELLOW IN TRACK SECTOR 12  DOUBLE YELLOW       NaN    1
5      Flag                                                                             

In [ ]:
import glob

laps_files = glob.glob("../data/raw/laps/*.csv")
print(f"total laps files: {len(laps_files)}")

status_samples = []
for f in laps_files:
    df = pd.read_csv(f, usecols=["TrackStatus"])
    status_samples.append(df)

track_status_all = pd.concat(status_samples, ignore_index=True)
print(f"total rows: {len(track_status_all)}")
print()
print(track_status_all["TrackStatus"].value_counts())


total laps files: 172
total rows: 188482

TrackStatus
1.0        165863
12.0         6116
4.0          5756
41.0         1976
124.0        1411
            ...  
1246.0          1
512.0           1
16724.0         1
651.0           1
1245.0          1
Name: count, Length: 48, dtype: int64


In [ ]:
code_labels = {
    "1": "clear",
    "2": "yellow",
    "3": "unknown",
    "4": "safety_car",
    "5": "red_flag",
    "6": "vsc_deployed",
    "7": "vsc_ending",
}

def decode_status(value):
    digits = str(int(value))
    return [code_labels.get(d, f"unrecognized_{d}") for d in digits]

track_status_all["status_codes"] = track_status_all["TrackStatus"].apply(decode_status)
track_status_all["is_green_only"] = track_status_all["TrackStatus"] == 1.0

green = track_status_all["is_green_only"].sum()
affected = (~track_status_all["is_green_only"]).sum()

print(f"pure green flag laps: {green}")
print(f"laps with any non-green status: {affected}")
print(f"percentage affected: {affected / len(track_status_all) * 100:.1f}%")

ValueError: cannot convert float NaN to integer

In [ ]:
print(f"null TrackStatus: {track_status_all['TrackStatus'].isna().sum()}")

null TrackStatus: 1340


In [ ]:
code_labels = {
    "1": "clear",
    "2": "yellow",
    "3": "unknown",
    "4": "safety_car",
    "5": "red_flag",
    "6": "vsc_deployed",
    "7": "vsc_ending",
}

def decode_status(value):
    if pd.isna(value):
        return []
    digits = str(int(value))
    return [code_labels.get(d, f"unrecognized_{d}") for d in digits]

track_status_all["status_codes"] = track_status_all["TrackStatus"].apply(decode_status)
track_status_all["is_green_only"] = track_status_all["TrackStatus"] == 1.0

green = track_status_all["is_green_only"].sum()
affected = (~track_status_all["is_green_only"]).sum()

print(f"pure green flag laps: {green}")
print(f"laps with any non-green status: {affected}")
print(f"percentage affected: {affected / len(track_status_all) * 100:.1f}%")

pure green flag laps: 165863
laps with any non-green status: 22619
percentage affected: 12.0%


In [ ]:
null_rows = track_status_all[track_status_all["TrackStatus"].isna()]
print(f"total null rows: {len(null_rows)}")

total null rows: 1340


In [ ]:
status_samples = []
for f in laps_files:
    df = pd.read_csv(f, usecols=["Driver", "LapNumber", "TrackStatus", "FastF1Generated", "IsAccurate"])
    df["source_file"] = f
    status_samples.append(df)

track_status_all = pd.concat(status_samples, ignore_index=True)

null_rows = track_status_all[track_status_all["TrackStatus"].isna()]
print(null_rows["FastF1Generated"].value_counts())
print()
print(null_rows["IsAccurate"].value_counts())

FastF1Generated
False    1339
True        1
Name: count, dtype: int64

IsAccurate
False    1340
Name: count, dtype: int64


### TrackStatus coverage (tier two)
- 165863 out of 188482 laps are pure green flag (TrackStatus == 1)
- 22619 laps, about 12%, had at least one non-green event during the lap
  (yellow, safety car, red flag, VSC deployed or ending)
- 1340 laps have null TrackStatus, all of which also have IsAccurate == False
  from FastF1 itself, a known upstream limitation, not a pipeline gap
- Treatment: any feature using TrackStatus should also check IsAccurate,
  and exclude non-green laps before computing tyre degradation

In [ ]:
laps = dfs["fact_laps"].sort_values(["driver_id", "race_id", "lap_number"]).copy()

laps["next_pit_out_seconds"] = laps.groupby(["driver_id", "race_id"])["pit_out_seconds"].shift(-1)
laps["next_lap_number"] = laps.groupby(["driver_id", "race_id"])["lap_number"].shift(-1)

same_row = laps[laps["pit_in_seconds"].notna() & laps["pit_out_seconds"].notna()].copy()
same_row["pit_stop_duration"] = same_row["pit_out_seconds"] - same_row["pit_in_seconds"]

split_row = laps[laps["pit_in_seconds"].notna() & laps["pit_out_seconds"].isna()].copy()
split_row = split_row[split_row["next_lap_number"] == split_row["lap_number"] + 1]
split_row["pit_stop_duration"] = split_row["next_pit_out_seconds"] - split_row["pit_in_seconds"]

pit_stops = pd.concat([
    same_row[["race_id", "driver_id", "lap_number", "pit_stop_duration"]],
    split_row[["race_id", "driver_id", "lap_number", "pit_stop_duration"]]
]).reset_index(drop=True)

print(pit_stops["pit_stop_duration"].describe())
print(f"negative durations: {(pit_stops['pit_stop_duration'] < 0).sum()}")
print(f"durations over 120 seconds: {(pit_stops['pit_stop_duration'] > 120).sum()}")

count    5936.000000
mean      125.019212
std       596.065958
min      -188.979000
25%        21.751500
50%        23.584500
75%        27.531000
max      7610.050000
Name: pit_stop_duration, dtype: float64
negative durations: 321
durations over 120 seconds: 307


In [ ]:
laps_per_race = laps.groupby("race_id")["lap_number"].max()
expected_range = laps_per_race.quantile([0.1, 0.5, 0.9])
print(expected_range)
print()
print(f"races with unusually low lap counts: {(laps_per_race < 30).sum()}")
print(laps_per_race[laps_per_race < 30])

0.1    51.0
0.5    58.0
0.9    71.0
Name: lap_number, dtype: float64

races with unusually low lap counts: 2
race_id
29     3
79    29
Name: lap_number, dtype: int64


In [ ]:
quali_files = glob.glob("../data/raw/qualifying/*.csv")
print(f"total qualifying files: {len(quali_files)}")

sample_quali = pd.read_csv(quali_files[0])
print(sample_quali.columns.tolist())
print(sample_quali.isnull().sum())

total qualifying files: 173
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']
Time                 0
Driver               0
DriverNumber         0
LapTime             99
LapNumber            0
                  ... 
Position           327
Deleted              0
DeletedReason      321
FastF1Generated      0
IsAccurate           0
Length: 31, dtype: int64


In [ ]:
era_counts = dfs["dim_race"].groupby("season_era")["race_id"].nunique()
print(era_counts)

era_lap_check = laps.merge(dfs["dim_race"][["race_id", "season_era"]], on="race_id")
print(era_lap_check.groupby("season_era")["lap_seconds"].mean())

season_era
covid          17
post_covid    114
pre_covid      42
Name: race_id, dtype: int64
season_era
covid         89.212175
post_covid    91.492166
pre_covid     92.333663
Name: lap_seconds, dtype: float64


In [ ]:
pit_stops_clean = pit_stops[(pit_stops["pit_stop_duration"] > 0) & (pit_stops["pit_stop_duration"] <= 100)]
print(f"clean pit stops: {len(pit_stops_clean)} out of {len(pit_stops)}")

clean pit stops: 5308 out of 5937


In [ ]:
race_29_info = dfs["dim_race"][dfs["dim_race"]["race_id"] == 29]
print(race_29_info[["race_id", "year", "round", "race_name"]])

    race_id  year  round           race_name
28       29  2021     12  Belgian Grand Prix


## Tier Two Findings, Domain-Specific Racing Analytics

### TrackStatus coverage
- 165863 of 188482 laps pure green flag, 22619 laps (12%) had a non-green event
- 1340 null TrackStatus rows, all tied to FastF1's own IsAccurate == False flag,
  upstream limitation, not a pipeline gap

### Pit stop duration at scale
- Raw duration calc produced 321 negative and 307 over 120 seconds, both invalid
- Filtered to 0-100 second range, 5308 of 5937 pit stops retained as clean
- Root cause of remaining bad rows not fully diagnosed, likely double stacked
  stops under safety car, deprioritized given time constraints

### Missing data patterns
- race_id 29, 2021 Belgian Grand Prix, only 3 laps recorded, real event,
  session abandoned due to weather after minimal running behind safety car
- race_id 79, 29 laps, short circuit, normal, no issue

### Qualifying data
- LapTime and Position nulls expected, tied to non-timed laps and drivers
  eliminated in Q1/Q2, no action needed

### Season era
- Lap pace varies slightly by era (89-92 seconds average), consistent with
  known regulation and circuit differences, no anomaly

### Race control confirmation
- 2024 Monaco Grand Prix (race_id 157) red flag and standing restart on lap 1
  confirmed via race control messages, explains tier one's extreme lap anomaly

In [ ]:
quali_samples = []
for f in quali_files:
    df = pd.read_csv(f, usecols=["Driver", "LapNumber", "LapTime", "Position", "Deleted", "DeletedReason", "IsAccurate"])
    df["source_file"] = f
    quali_samples.append(df)

quali_all = pd.concat(quali_samples, ignore_index=True)

print(f"total qualifying laps across all races: {len(quali_all)}")
print()
print(f"null LapTime: {quali_all['LapTime'].isna().sum()} ({quali_all['LapTime'].isna().mean()*100:.1f}%)")
print(f"null Position: {quali_all['Position'].isna().sum()} ({quali_all['Position'].isna().mean()*100:.1f}%)")
print()
print(f"deleted laps: {quali_all['Deleted'].sum()}")
print(quali_all["DeletedReason"].value_counts())

total qualifying laps across all races: 53690

null LapTime: 17114 (31.9%)
null Position: 53690 (100.0%)

deleted laps: 627
DeletedReason
TRACK LIMITS AT TURN 1                   38
TRACK LIMITS AT TURN 9                   25
TRACK LIMITS AT TURN 4                   16
TRACK LIMITS AT TURN 10                  13
TRACK LIMITS AT TURN 11                  12
                                         ..
TRACK LIMITS AT TURN 8 LAP 7              1
TRACK LIMITS AT TURN 2 LAP 8 7:49:50      1
TRACK LIMITS AT TURN 4 LAP 18 8:23:08     1
TRACK LIMITS AT TURN 4 LAP 5 7:36:52      1
TRACK LIMITS AT TURN 20 LAP 6             1
Name: count, Length: 256, dtype: int64


### Qualifying data quality
- 53690 total qualifying laps across 173 races
- Position is 100% null, confirmed expected per FastF1 docs, only populated
  for Race and Sprint sessions, not Qualifying
- LapTime null on 31.9% of laps, expected, includes out-laps, in-laps, and
  aborted attempts with no timed result
- 627 laps deleted, majority for track limits violations, real signal,
  drivers push track limits harder in qualifying
- Gap: no Q1/Q2/Q3 segment marker in the raw pull, FastF1's
  split_qualifying_sessions() was never called during ingestion, would need
  a small addition to pull_data.py if a future model needs segment-level
  qualifying pace